# Gold · KPI Alertas

## Propósito

Identificar los pedidos que cumplen criterios de riesgo crítico y entregarlos como snapshot periódico.

## Flujo

1. Seleccionar los pedidos con riesgo de entrega no validado o con sospecha de fraude.
2. Asignar el motivo de la alerta.
3. Guardar el resultado como snapshot del corte, marcando el más reciente.

## Destino

Capa Gold.

**Tabla:** `prod_dataco.gld_dataco.kpi_alertas`

**Grano:** un registro por pedido en alerta, por corte

## Motivos de alerta

* Riesgo no validado
* Fraude sospechoso
* Riesgo no validado y fraude sospechoso

## Historial

* `fecha_snapshot` corresponde a la fecha de fin del corte.
* `es_ultimo_snapshot` marca el snapshot más reciente.
* El proceso es idempotente por corte.

## Resultado esperado

La lista de pedidos en alerta lista para la pestaña Alertas del Excel y el conteo del correo.

In [0]:
# ============================================================
# GLD_KPI_ALERTAS — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog", "prod_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "gld_dataco", "Esquema Gold")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

GOLD_ORDER_SUMMARY_FQN = f"{gold_catalog}.{gold_schema}.order_summary"
GOLD_KPI_ALERTAS_FQN = f"{gold_catalog}.{gold_schema}.kpi_alertas"

print(f"✓ Origen: {GOLD_ORDER_SUMMARY_FQN}")
print(f"✓ Destino: {GOLD_KPI_ALERTAS_FQN}")

In [0]:
# ============================================================
# GLD_KPI_ALERTAS — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control", key="id_ejecucion", default="", debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_kpi_alertas"
CAPA = "gold"

spark.sql(f"""
INSERT INTO prod_dataco.metadata.etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_KPI_ALERTAS — Celda 3: Pedidos en riesgo o inconsistentes
# ============================================================

from pyspark.sql import functions as F

df_base = spark.table(GOLD_ORDER_SUMMARY_FQN)

df_alertas = (
    df_base
    .filter((F.col("riesgo_validado") == False) | (F.col("es_sospechoso_fraude") == True))
    .withColumn(
        "motivo_alerta",
        F.when(
            (F.col("riesgo_validado") == False) & (F.col("es_sospechoso_fraude") == True),
            F.lit("RIESGO_NO_VALIDADO_Y_FRAUDE_SOSPECHOSO")
        )
        .when(F.col("riesgo_validado") == False, F.lit("RIESGO_NO_VALIDADO"))
        .otherwise(F.lit("FRAUDE_SOSPECHOSO"))
    )
    .select(
        "order_id", "market", "order_region", "customer_segment",
        "order_status", "delivery_status",
        "late_delivery_risk_original", "late_delivery_risk_calculado",
        "riesgo_validado", "es_sospechoso_fraude", "motivo_alerta",
        "total_sales", "total_profit", "fecha_fin_corte"
    )
)

print(f"✓ Total de pedidos en alerta: {df_alertas.count()}")

display(df_alertas.limit(10))

In [0]:
# ============================================================
# GLD_KPI_ALERTAS — Celda 4: Snapshot periódico (historial)
# ============================================================

from delta.tables import DeltaTable

corte_actual = df_base.select(F.max("fecha_fin_corte").alias("m")).collect()[0]["m"]

df_alertas_final = df_alertas.withColumn("fecha_snapshot", F.lit(corte_actual))

if spark.catalog.tableExists(GOLD_KPI_ALERTAS_FQN):
    ya_existe = (
        spark.table(GOLD_KPI_ALERTAS_FQN)
        .filter(F.col("fecha_snapshot") == corte_actual)
        .limit(1).count() > 0
    )

    if not ya_existe:
        delta_tbl = DeltaTable.forName(spark, GOLD_KPI_ALERTAS_FQN)
        delta_tbl.update(
            condition=F.col("es_ultimo_snapshot") == True,
            set={"es_ultimo_snapshot": F.lit(False)}
        )
        (
            df_alertas_final.withColumn("es_ultimo_snapshot", F.lit(True))
            .write.format("delta").mode("append").saveAsTable(GOLD_KPI_ALERTAS_FQN)
        )
        print(f"✓ Nuevo snapshot agregado para el corte {corte_actual}")
    else:
        print(f"⚠️ El snapshot del corte {corte_actual} ya existía — no se duplicó (idempotente)")
else:
    (
        df_alertas_final.withColumn("es_ultimo_snapshot", F.lit(True))
        .write.format("delta").mode("overwrite").saveAsTable(GOLD_KPI_ALERTAS_FQN)
    )
    print(f"✓ Tabla {GOLD_KPI_ALERTAS_FQN} creada con el primer snapshot")

In [0]:
# ============================================================
# GLD_KPI_ALERTAS — Celda 5: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE prod_dataco.metadata.etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")